# GraphRAG End-to-End Pipeline

**Goal:** text → chunks → extracted graph → Neo4j → graph retrieval → answer.


## 1. Libraries and what they do

| Import / component | Type | Purpose |
|---|---|---|
| `Document` | LangChain library | Stores text plus metadata |
| `RecursiveCharacterTextSplitter` | LangChain library | Splits long text into overlapping chunks |
| `ChatOpenAI` | LangChain integration | Calls an OpenAI-compatible chat model |
| `LLMGraphTransformer` | LangChain experimental library | Asks the LLM to extract entities and relationships |
| `Neo4jGraph` | LangChain Neo4j integration | Writes graph documents to Neo4j and runs Cypher |
| `os`, `getpass` | Python built-in | Reads configuration safely |
| Functions defined in this notebook | Manual | Show the mechanics explicitly |

**Setup once in your environment:** install `langchain-core`, `langchain-text-splitters`, `langchain-openai`, `langchain-experimental`, and `langchain-neo4j`. Configure `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` in your environment. If your LLM gateway needs a custom URL, also configure `OPENAI_BASE_URL`.

In [2]:
import os
from getpass import getpass

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph

print("Imports loaded.")
print("Document = text + metadata; splitter = chunking; ChatOpenAI = model client;")
print("LLMGraphTransformer = graph extraction; Neo4jGraph = graph database connection.")


/home/ekkel-ai/momina/projects/graphrag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_17481/3218405835.py:7: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


Imports loaded.
Document = text + metadata; splitter = chunking; ChatOpenAI = model client;
LLMGraphTransformer = graph extraction; Neo4jGraph = graph database connection.


## 2. Configure the model and Neo4j

**Input:** credentials and model settings from environment variables.  
**Output:** a model client and a Neo4j connection.  
**Type:** library clients, configured by this notebook.

In [4]:
from dotenv import load_dotenv, find_dotenv
import os

env_path = find_dotenv(usecwd=True)

if not env_path:
    raise FileNotFoundError("Could not find .env file")

load_dotenv(env_path, override=True)

print("Loaded environment file:", env_path)
print("Neo4j URI:", os.getenv("NEO4J_URI"))
print("Neo4j username:", os.getenv("NEO4J_USERNAME"))

Loaded environment file: /home/ekkel-ai/momina/projects/graphrag/.env
Neo4j URI: neo4j+s://cb96a564.databases.neo4j.io
Neo4j username: cb96a564


In [12]:
api_key = os.getenv("LLM_API_KEY")
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

missing = [
    name for name, value in {
        "LLM_API_KEY": api_key,
        "NEO4J_URI": neo4j_uri,
        "NEO4J_USERNAME": neo4j_username,
        "NEO4J_PASSWORD": neo4j_password,
    }.items() if not value
]
if missing:
    raise ValueError("Set these environment variables before continuing: " + ", ".join(missing))

model_name = os.getenv("LLM_MODEL", "gpt-4o-mini")
llm_kwargs = {"model": model_name, "api_key": api_key, "temperature": 0}
if os.getenv("LLM_BASE_URL"):
    llm_kwargs["base_url"] = os.getenv("LLM_BASE_URL")

llm = ChatOpenAI(**llm_kwargs)
graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
)

print(f"LLM client configured: {model_name}")
print(f"Neo4j connection configured for: {neo4j_uri}")
print("Credentials are not displayed.")


LLM client configured: openai/gpt-oss-120b
Neo4j connection configured for: neo4j+s://cb96a564.databases.neo4j.io
Credentials are not displayed.


## Unstructured Data

## 3. Sample paragraph

**Input:** raw text.  
**Output:** one LangChain `Document`.  
**Type:** `Document` is library-provided; the story is our example data.

In [6]:
# Source document
story = (
    "Momna lives in Willow Village. "
    "Momna grows vegetables in a community garden. "
    "Anfaal teaches Momna how to collect rainwater. "
    "Momna uses the rainwater to water the garden. "
    "The garden provides vegetables to the village school. "
    "Zainab is the school cook. "
    "Zainab prepares lunch using those vegetables."
)

source_doc = Document(
    page_content=story,
    metadata={"source": "tiny_garden_story", "document_id": "tiny_garden_demo"},
)

print("SOURCE TEXT:")
print(source_doc.page_content)
print("\nMETADATA:", source_doc.metadata)


SOURCE TEXT:
Momna lives in Willow Village. Momna grows vegetables in a community garden. Anfaal teaches Momna how to collect rainwater. Momna uses the rainwater to water the garden. The garden provides vegetables to the village school. Zainab is the school cook. Zainab prepares lunch using those vegetables.

METADATA: {'source': 'tiny_garden_story', 'document_id': 'tiny_garden_demo'}


## 4. Split text into chunks

**Input:** a list of `Document` objects, chunk size, overlap.  
**Output:** smaller `Document` objects.  
**Type:** `RecursiveCharacterTextSplitter` is a library class; `chunk_documents()` below is a manual wrapper.

Chunking helps process longer files in manageable pieces. Overlap keeps a little context between neighboring chunks.

In [8]:
# manual chunking function using the library splitter
def chunk_documents(documents, chunk_size=180, chunk_overlap=60):
    """Input: Documents. Output: chunk Documents with copied metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    return splitter.split_documents(documents)

chunks = chunk_documents([source_doc], chunk_size=180, chunk_overlap=60)

print(f"Input documents: 1 | Output chunks: {len(chunks)}")
for i, chunk in enumerate(chunks, start=1):
    print(f"\nCHUNK {i} | metadata={chunk.metadata}")
    print(chunk.page_content)


Input documents: 1 | Output chunks: 2

CHUNK 1 | metadata={'source': 'tiny_garden_story', 'document_id': 'tiny_garden_demo', 'start_index': 0}
Momna lives in Willow Village. Momna grows vegetables in a community garden. Anfaal teaches Momna how to collect rainwater. Momna uses the rainwater to water the garden. The garden

CHUNK 2 | metadata={'source': 'tiny_garden_story', 'document_id': 'tiny_garden_demo', 'start_index': 124}
Momna uses the rainwater to water the garden. The garden provides vegetables to the village school. Zainab is the school cook. Zainab prepares lunch using those vegetables.


## 5. Turn a chunk into a graph document

**Input:** one text chunk and the configured LLM.  
**Output:** graph nodes and relationships extracted from that chunk.  
**Type:** `LLMGraphTransformer` and `convert_to_graph_documents()` are library-provided; the extraction itself is performed by the model.

This is the important transformation: plain sentences become structured facts. Exact node names and relationship types can vary with the model.

In [9]:
# extract graph structure from the first chunk
transformer = LLMGraphTransformer(llm=llm)

sample_graph_documents = transformer.convert_to_graph_documents([chunks[0]])

print(f"Input: 1 chunk | Output graph documents: {len(sample_graph_documents)}")
for doc_index, graph_doc in enumerate(sample_graph_documents, start=1):
    print(f"\nGRAPH DOCUMENT {doc_index}")
    print("NODES:")
    for node in graph_doc.nodes:
        print({"id": node.id, "type": node.type, "properties": node.properties})
    print("RELATIONSHIPS:")
    for rel in graph_doc.relationships:
        print({
            "source": rel.source.id,
            "type": rel.type,
            "target": rel.target.id,
            "properties": rel.properties,
        })


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Input: 1 chunk | Output graph documents: 1

GRAPH DOCUMENT 1
NODES:
{'id': 'Momna', 'type': 'Person', 'properties': {}}
{'id': 'Willow Village', 'type': 'Location', 'properties': {}}
{'id': 'Community Garden', 'type': 'Garden', 'properties': {}}
{'id': 'Anfaal', 'type': 'Person', 'properties': {}}
{'id': 'Rainwater', 'type': 'Resource', 'properties': {}}
RELATIONSHIPS:
{'source': 'Momna', 'type': 'LIVES_IN', 'target': 'Willow Village', 'properties': {}}
{'source': 'Momna', 'type': 'GROWS_IN', 'target': 'Community Garden', 'properties': {}}
{'source': 'Anfaal', 'type': 'TEACHES', 'target': 'Momna', 'properties': {}}
{'source': 'Momna', 'type': 'USES', 'target': 'Rainwater', 'properties': {}}
{'source': 'Momna', 'type': 'WATERS', 'target': 'Community Garden', 'properties': {}}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 6. Inspect the conversion function's input and output

`convert_to_graph_documents()` **does not write to Neo4j**. It converts text into an in-memory graph representation.

In [10]:
# inspect extracted structure
all_graph_documents = transformer.convert_to_graph_documents(chunks)

node_total = sum(len(gd.nodes) for gd in all_graph_documents)
relationship_total = sum(len(gd.relationships) for gd in all_graph_documents)

print(f"Input chunks: {len(chunks)}")
print(f"Output graph documents: {len(all_graph_documents)}")
print(f"Extracted nodes: {node_total}")
print(f"Extracted relationships: {relationship_total}")
print("No database write has happened in this cell.")


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Input chunks: 2
Output graph documents: 2
Extracted nodes: 12
Extracted relationships: 12
No database write has happened in this cell.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 7. Add graph documents to Neo4j

**Input:** graph documents, plus source/document metadata.  
**Output:** nodes and relationships stored in Neo4j.  
**Type:** `graph.add_graph_documents()` is library-provided; `add_story_to_graph()` is a manual wrapper.

This demo tags extracted nodes and relationships with `document_id` so the data can be inspected later. Use a development database for this demo.

In [68]:
def add_story_to_graph(graph_documents, document_id):
    """Input: graph documents + ID. Output: count of submitted nodes/relationships."""
    submitted_nodes = 0
    submitted_relationships = 0

    for graph_doc in graph_documents:
        for node in graph_doc.nodes:
            node.properties["document_id"] = document_id
        for relationship in graph_doc.relationships:
            relationship.properties["document_id"] = document_id

        submitted_nodes += len(graph_doc.nodes)
        submitted_relationships += len(graph_doc.relationships)

    graph.add_graph_documents(graph_documents)
    return {
        "graph_documents": len(graph_documents),
        "nodes_submitted": submitted_nodes,
        "relationships_submitted": submitted_relationships,
        "document_id": document_id,
    }

write_result = add_story_to_graph(all_graph_documents, "tiny_garden_demo")
print("Write result:", write_result)
print("Neo4j now has the extracted graph data (verify it in the next cell).")


Write result: {'graph_documents': 2, 'nodes_submitted': 10, 'relationships_submitted': 8, 'document_id': 'tiny_garden_demo'}
Neo4j now has the extracted graph data (verify it in the next cell).


## 8. Verify what is stored

Cypher is Neo4j's query language. This cell runs read-only queries and prints the stored nodes and relationships.

**Input:** Cypher text. **Output:** rows returned from Neo4j.  
**Type:** Cypher is the query language; these query strings are written manually.

In [69]:
node_rows = graph.query(
    "MATCH (n) WHERE n.id = $doc_id "
    "RETURN labels(n) AS labels, n.id AS id, properties(n) AS properties LIMIT 30",
    params={"doc_id": "tiny_garden_demo"},
)
relationship_rows = graph.query(
    "MATCH (a)-[r]->(b) "
    "WHERE r.document_id = $doc_id "
    "RETURN a.id AS source, type(r) AS relationship, b.id AS target, properties(r) AS properties LIMIT 30",
    params={"doc_id": "tiny_garden_demo"},
)

print(f"Stored node rows returned: {len(node_rows)}")
for row in node_rows:
    print(row)

print(f"\nStored relationship rows returned: {len(relationship_rows)}")
for row in relationship_rows:
    print(row)


Stored node rows returned: 0

Stored relationship rows returned: 8
{'source': 'Momna', 'relationship': 'GROWS', 'target': 'Community Garden', 'properties': {'document_id': 'tiny_garden_demo'}}
{'source': 'Momna', 'relationship': 'USES', 'target': 'Rainwater', 'properties': {'document_id': 'tiny_garden_demo'}}
{'source': 'Momna', 'relationship': 'WATERS', 'target': 'Community Garden', 'properties': {'document_id': 'tiny_garden_demo'}}
{'source': 'Momna', 'relationship': 'RESIDES', 'target': 'Willow Village', 'properties': {'document_id': 'tiny_garden_demo'}}
{'source': 'Anfaal', 'relationship': 'TEACHES', 'target': 'Momna', 'properties': {'document_id': 'tiny_garden_demo'}}
{'source': 'Garden', 'relationship': 'PROVIDES', 'target': 'Village School', 'properties': {'document_id': 'tiny_garden_demo'}}
{'source': 'Zainab', 'relationship': 'PREPARES', 'target': 'Lunch', 'properties': {'document_id': 'tiny_garden_demo'}}
{'source': 'Lunch', 'relationship': 'USES', 'target': 'Vegetables', 'pr

## 10. Retrieve relevant graph facts for a question

**Input:** entity name and hop limit. **Output:** graph facts.  
**Type:** `local_graph_retrieval()` is manual; the Cypher execution is library-provided.

In [71]:
def local_graph_retrieval(entity_name, max_hops=2, limit=20):
    """Input: entity + hop depth. Output: rows describing nearby graph facts."""
    if max_hops not in (1, 2, 3):
        raise ValueError("max_hops must be 1, 2, or 3.")

    query = f'''
    MATCH path = (start)-[*1..{max_hops}]-(other)
    WHERE toLower(toString(start.id)) CONTAINS toLower($entity_name)
      AND any(n IN nodes(path) WHERE n.document_id = $doc_id)
    RETURN [n IN nodes(path) | coalesce(toString(n.id), labels(n)[0])] AS path_nodes,
           [r IN relationships(path) | type(r)] AS path_relationships
    LIMIT {int(limit)}
    '''
    return graph.query(query, params={
        "entity_name": entity_name,
        "doc_id": "tiny_garden_demo",
    })

retrieved_facts = local_graph_retrieval("Anfaal", max_hops=3)
print(f"Retrieved {len(retrieved_facts)} graph paths about Anfaal.")
for fact in retrieved_facts:
    print(fact)


Retrieved 7 graph paths about Anfaal.
{'path_nodes': ['Anfaal', 'Momna'], 'path_relationships': ['TEACHES']}
{'path_nodes': ['Anfaal', 'Momna', 'Community Garden'], 'path_relationships': ['TEACHES', 'GROWS']}
{'path_nodes': ['Anfaal', 'Momna', 'Community Garden', 'Momna'], 'path_relationships': ['TEACHES', 'GROWS', 'WATERS']}
{'path_nodes': ['Anfaal', 'Momna', 'Rainwater'], 'path_relationships': ['TEACHES', 'USES']}
{'path_nodes': ['Anfaal', 'Momna', 'Community Garden'], 'path_relationships': ['TEACHES', 'WATERS']}
{'path_nodes': ['Anfaal', 'Momna', 'Community Garden', 'Momna'], 'path_relationships': ['TEACHES', 'WATERS', 'GROWS']}
{'path_nodes': ['Anfaal', 'Momna', 'Willow Village'], 'path_relationships': ['TEACHES', 'RESIDES']}


## 11. Use the retrieved facts to answer a question

The graph finds connected evidence; the LLM turns that evidence into a readable answer. The answer should be grounded in the retrieved paths rather than relying only on model memory.

**Input:** question + retrieved graph facts.  
**Output:** natural-language answer.  
**Type:** `answer_from_graph()` is manual; `llm.invoke()` is library-provided.

In [74]:
def answer_from_graph(question, facts):
    """Input: question and graph facts. Output: LLM answer grounded in those facts."""
    if not facts:
        return "No matching graph facts were found. Try a different entity or hop depth."

    prompt = f'''
Answer the question using only the graph facts below.
If the facts do not support an answer, say what is missing.
Keep the answer short and clear. First try to match the link between the all given facts.dont skip quickly.

Question: {question}

Graph facts:
{facts}
'''
    response = llm.invoke(prompt)
    return response.content

question = "what does anfaal teaches momna?"
answer = answer_from_graph(question, retrieved_facts)

print("QUESTION:", question)
print("\nRETRIEVED FACTS USED:", len(retrieved_facts))
print(retrieved_facts)
print("\nANSWER:")
print(answer)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


QUESTION: what does anfaal teaches momna?

RETRIEVED FACTS USED: 7
[{'path_nodes': ['Anfaal', 'Momna'], 'path_relationships': ['TEACHES']}, {'path_nodes': ['Anfaal', 'Momna', 'Community Garden'], 'path_relationships': ['TEACHES', 'GROWS']}, {'path_nodes': ['Anfaal', 'Momna', 'Community Garden', 'Momna'], 'path_relationships': ['TEACHES', 'GROWS', 'WATERS']}, {'path_nodes': ['Anfaal', 'Momna', 'Rainwater'], 'path_relationships': ['TEACHES', 'USES']}, {'path_nodes': ['Anfaal', 'Momna', 'Community Garden'], 'path_relationships': ['TEACHES', 'WATERS']}, {'path_nodes': ['Anfaal', 'Momna', 'Community Garden', 'Momna'], 'path_relationships': ['TEACHES', 'WATERS', 'GROWS']}, {'path_nodes': ['Anfaal', 'Momna', 'Willow Village'], 'path_relationships': ['TEACHES', 'RESIDES']}]

ANSWER:
Anfaal teaches Momna about the **Community Garden**, **Rainwater**, and **Willow Village**.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## test query

**Input:** question + entity + hop depth. **Output:** retrieved facts and an answer.

In [75]:
question_2 = "How does the garden provide food to the school?"
entity_2 = "garden"

facts_2 = local_graph_retrieval(entity_2, max_hops=3)
answer_2 = answer_from_graph(question_2, facts_2)

print("QUESTION:", question_2)
print(f"ENTITY: {entity_2} | HOP DEPTH: 3 | PATHS: {len(facts_2)}")
print("\nSAMPLE FACTS:")
for fact in facts_2[:5]:
    print(fact)
print("\nANSWER:")
print(answer_2)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


QUESTION: How does the garden provide food to the school?
ENTITY: garden | HOP DEPTH: 3 | PATHS: 11

SAMPLE FACTS:
{'path_nodes': ['Community Garden', 'Momna'], 'path_relationships': ['GROWS']}
{'path_nodes': ['Community Garden', 'Momna', 'Anfaal'], 'path_relationships': ['GROWS', 'TEACHES']}
{'path_nodes': ['Community Garden', 'Momna', 'Rainwater'], 'path_relationships': ['GROWS', 'USES']}
{'path_nodes': ['Community Garden', 'Momna', 'Community Garden'], 'path_relationships': ['GROWS', 'WATERS']}
{'path_nodes': ['Community Garden', 'Momna', 'Willow Village'], 'path_relationships': ['GROWS', 'RESIDES']}

ANSWER:
The garden **provides** (presumably food) to the Village School, as shown by the fact ‘Garden → PROVIDES → Village School’.  
*Missing:* an explicit statement that what is provided is food.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Structured data 

## load and prepare excel data

In [24]:
import pandas as pd
import re

xlsx_path = "data/raw/meezan.xlsx"

df = pd.read_excel(xlsx_path, sheet_name="Sheet1", header=1, dtype=str)

# Keep the original column names and values.
df.columns = [str(c).strip() for c in df.columns]
df = df.fillna("")

# Clean whitespace without converting phone numbers to integers.
for col in df.columns:
    df[col] = df[col].str.strip()

required = [
    "S.No", "Branch Code", "Branch Name",
    "OM Landline No", "BM Landline No",
    "Branch Landline Number", "City Name",
    "Area Name", "Province", "Branch Address"
]

assert all(col in df.columns for col in required), "Unexpected Excel columns!"
assert len(df.columns) == 10, "Expected all 10 original columns."


codes = df["Branch Code"].str.strip()
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Duplicate branch codes:", codes.duplicated().sum())
print("Empty branch codes:", (codes == "").sum())

display(df.head(3))

Rows: 1102
Columns: ['S.No', 'Branch Code', 'Branch Name', 'OM Landline No', 'BM Landline No', 'Branch Landline Number', 'City Name', 'Area Name', 'Province', 'Branch Address']
Duplicate branch codes: 0
Empty branch codes: 0


,S.No,Branch Code,Branch Name,OM Landline No,BM Landline No,Branch Landline Number,City Name,Area Name,Province,Branch Address
0,1,105,"Marriott Hotel, Karachi",021-35693166,021-5682682,021-5683491,Karachi,Clifton,Sindh,"Marriott Hotel, Abdullah Haroon Road, Karachi"
1,2,201,"Gulberg, Lahore",042-35755716\n(042-35777434),042-35870929,042-5879870-72,Lahore,Gulberg,Punjab,"60 Main Boulevard, Gulberg, Lahore"
2,3,301,"Jinnah Avenue (Blue Area), Islamabad",051-2206513,051-2277518,051-2276712-15,Islamabad,Islamabad East,Islamabad,"Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue ..."


## converting rows into graph ready records

In [77]:
def normalize_name(value):
    value = str(value).strip().lower()
    return re.sub(r"\s+", " ", value)

records = []

for row in df.to_dict(orient="records"):
    branch_code = row["Branch Code"].strip()

    if not branch_code:
        continue

    city = row["City Name"].strip()
    area = row["Area Name"].strip()
    province = row["Province"].strip()

    records.append({
        "branch_code": branch_code,
        "area": area,
        "area_key": f"{normalize_name(city)}|{normalize_name(area)}",
        "city": city,
        "city_key": normalize_name(city),
        "province": province,
        "province_key": normalize_name(province),
        "properties": row,  # the all 10 original Excel columns
        "dataset_id": "meezan_branches"
    })

print("Graph-ready records:", len(records))
print("Original columns preserved:", list(records[0]["properties"].keys()))
print("Example record:")
print(records[0])

Graph-ready records: 1102
Original columns preserved: ['S.No', 'Branch Code', 'Branch Name', 'OM Landline No', 'BM Landline No', 'Branch Landline Number', 'City Name', 'Area Name', 'Province', 'Branch Address']
Example record:
{'branch_code': '105', 'area': 'Clifton', 'area_key': 'karachi|clifton', 'city': 'Karachi', 'city_key': 'karachi', 'province': 'Sindh', 'province_key': 'sindh', 'properties': {'S.No': '1', 'Branch Code': '105', 'Branch Name': 'Marriott Hotel, Karachi', 'OM Landline No': '021-35693166', 'BM Landline No': '021-5682682', 'Branch Landline Number': '021-5683491', 'City Name': 'Karachi', 'Area Name': 'Clifton', 'Province': 'Sindh', 'Branch Address': 'Marriott Hotel, Abdullah Haroon Road, Karachi'}, 'dataset_id': 'meezan_branches'}


## insert structured data into Neo4j

In [78]:
#a safety check that  do not merge records under duplicate branch codes.
branch_codes = [r["branch_code"] for r in records]
if len(branch_codes) != len(set(branch_codes)):
    raise ValueError("Duplicate branch codes found. Resolve them before ingestion.")

graph.query("""
UNWIND $rows AS row

MERGE (b:Branch {
    dataset_id: row.dataset_id,
    branch_code: row.branch_code
})
SET b += row.properties

MERGE (a:Area {
    dataset_id: row.dataset_id,
    name_key: row.area_key
})
SET a.name = row.area,
    a.city_name = row.city

MERGE (c:City {
    dataset_id: row.dataset_id,
    name_key: row.city_key
})
SET c.name = row.city

MERGE (p:Province {
    dataset_id: row.dataset_id,
    name_key: row.province_key
})
SET p.name = row.province

MERGE (b)-[:LOCATED_IN]->(a)
MERGE (a)-[:WITHIN_CITY]->(c)
MERGE (c)-[:WITHIN_PROVINCE]->(p)
""", {"rows": records})

print("Structured branch data ingested.")

Structured branch data ingested.


In [37]:
result = graph.query("""
MATCH (b:Branch {dataset_id: "meezan_branches"})
RETURN count(b) AS total_branches
""")
print(result)

result = graph.query("""
MATCH (b:Branch {dataset_id: "meezan_branches"})
RETURN b.`Branch Code` AS branch_code,
       b.`Branch Name` AS branch_name,
       b.`BM Landline No` AS manager_phone,
       b.`Branch Address` AS address
LIMIT 5
""")
print(result)

[{'total_branches': 1102}]
[{'branch_code': '105', 'branch_name': 'Marriott Hotel, Karachi', 'manager_phone': '021-5682682', 'address': 'Marriott Hotel, Abdullah Haroon Road, Karachi'}, {'branch_code': '201', 'branch_name': 'Gulberg, Lahore', 'manager_phone': '042-35870929', 'address': '60 Main Boulevard, Gulberg, Lahore'}, {'branch_code': '301', 'branch_name': 'Jinnah Avenue (Blue Area), Islamabad', 'manager_phone': '051-2277518', 'address': 'Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue Area, Islamabad.'}, {'branch_code': '101', 'branch_name': 'PNSC (Lalazar), Karachi', 'manager_phone': '021-35636252', 'address': '37-A, Lalazar Area, Off Moulvi Tamizuddin Khan Road, Karachi'}, {'branch_code': '401', 'branch_name': 'Sereena Hotel, Faisalabad', 'manager_phone': '041-2601225', 'address': 'Shop No.G-737 to G-740, G-751 & G-753, Bazar Court, Faisalabad Serena Hotel, Club Road, Faisalabad'}]


## inspect the stored data in neo4j

In [11]:
print("NODE COUNTS")
print(graph.query("""
MATCH (n {dataset_id: "meezan_branches"})
RETURN labels(n) AS node_types, count(n) AS total
ORDER BY total DESC
"""))

print("\nRELATIONSHIPS")
print(graph.query("""
MATCH (a {dataset_id: "meezan_branches"})
      -[r]->
      (b {dataset_id: "meezan_branches"})
RETURN labels(a) AS from_type,
       type(r) AS relationship,
       labels(b) AS to_type,
       count(*) AS total
"""))

print("\nONE COMPLETE BRANCH RECORD")
print(graph.query("""
MATCH (b:Branch {dataset_id: "meezan_branches"})
RETURN properties(b) AS branch_properties
LIMIT 1
"""))

NODE COUNTS
[{'node_types': ['Branch'], 'total': 1102}, {'node_types': ['Area'], 'total': 396}, {'node_types': ['City'], 'total': 328}, {'node_types': ['Province'], 'total': 7}]

RELATIONSHIPS
[{'from_type': ['Branch'], 'relationship': 'LOCATED_IN', 'to_type': ['Area'], 'total': 1102}, {'from_type': ['Area'], 'relationship': 'WITHIN_CITY', 'to_type': ['City'], 'total': 396}, {'from_type': ['City'], 'relationship': 'WITHIN_PROVINCE', 'to_type': ['Province'], 'total': 329}]

ONE COMPLETE BRANCH RECORD
[{'branch_properties': {'branch_code': '105', 'BM Landline No': '021-5682682', 'Branch Address': 'Marriott Hotel, Abdullah Haroon Road, Karachi', 'Branch Code': '105', 'dataset_id': 'meezan_branches', 'Area Name': 'Clifton', 'Branch Landline Number': '021-5683491', 'City Name': 'Karachi', 'S.No': '1', 'Branch Name': 'Marriott Hotel, Karachi', 'Province': 'Sindh', 'OM Landline No': '021-35693166'}}]


## Local Reterival

In [9]:
#Direct local retrieval

question = "What is the BM landline for Gulberg, Lahore?"

retrieved = graph.query("""
MATCH (b:Branch {dataset_id: "meezan_branches"})
      -[:LOCATED_IN]->(a:Area)
      -[:WITHIN_CITY]->(c:City)
WHERE toLower(a.name) CONTAINS toLower($area)
  AND toLower(c.name) CONTAINS toLower($city)
RETURN properties(b) AS branch,
       a.name AS matched_area,
       c.name AS matched_city
LIMIT 10
""", {
    "area": "Gulberg",
    "city": "Lahore"
})

print("USER QUESTION:", question)
print("\nRETRIEVED RECORDS:")
for item in retrieved:
    print(item)

USER QUESTION: What is the BM landline for Gulberg, Lahore?

RETRIEVED RECORDS:
{'branch': {'branch_code': '201', 'BM Landline No': '042-35870929', 'Branch Address': '60 Main Boulevard, Gulberg, Lahore', 'Branch Code': '201', 'dataset_id': 'meezan_branches', 'Area Name': 'Gulberg', 'Branch Landline Number': '042-5879870-72', 'City Name': 'Lahore', 'S.No': '2', 'Branch Name': 'Gulberg, Lahore', 'Province': 'Punjab', 'OM Landline No': '042-35755716\n(042-35777434)'}, 'matched_area': 'Gulberg', 'matched_city': 'Lahore'}
{'branch': {'branch_code': '216', 'BM Landline No': '042-6619786', 'Branch Address': 'Plot # 72, Commercial Area, Cavalry Ground Lahore Cantt.', 'Branch Code': '216', 'dataset_id': 'meezan_branches', 'Area Name': 'Gulberg', 'Branch Landline Number': '042-6619780-83', 'City Name': 'Lahore', 'S.No': '84', 'Branch Name': 'Cavalry Ground, Lahore', 'Province': 'Punjab', 'OM Landline No': '042-36619787'}, 'matched_area': 'Gulberg', 'matched_city': 'Lahore'}
{'branch': {'branch_c

## Query pipeline 

### 1. get the graph schema
- Input: existing Neo4j database.
- Output: Node labels, properties with observed value types, and relationship types.

In [13]:
import json

# Get node labels and their properties.
node_rows = graph.query("""
MATCH (n {dataset_id: "meezan_branches"})
UNWIND labels(n) AS label
UNWIND keys(n) AS property
RETURN DISTINCT label, property
ORDER BY label, property
""")

# Get relationship types and their paths.
relationship_rows = graph.query("""
MATCH (a {dataset_id: "meezan_branches"})
      -[r]->
      (b {dataset_id: "meezan_branches"})
RETURN DISTINCT labels(a) AS source,
       type(r) AS relationship,
       labels(b) AS target
ORDER BY relationship
""")

schema = {"nodes": {}, "relationships": []}

for row in node_rows:
    label = row["label"]
    prop = row["property"]
    schema["nodes"].setdefault(label, {"properties": {}})
    schema["nodes"][label]["properties"][prop] = "observed type pending"

# Sample actual values to estimate property types.
for label, info in schema["nodes"].items():
    sample = graph.query(f"""
    MATCH (n:{label} {{dataset_id: "meezan_branches"}})
    RETURN properties(n) AS props
    LIMIT 20
    """)

    for prop in info["properties"]:
        values = [
            row["props"].get(prop)
            for row in sample
            if row["props"].get(prop) is not None
        ]
        types = sorted({type(v).__name__ for v in values})
        info["properties"][prop] = (
            ", ".join(types) if types else "unknown"
        )

schema["relationships"] = relationship_rows

print("INPUT: Existing Neo4j graph")
print("\nOUTPUT: DISCOVERED SCHEMA")
print(json.dumps(schema, indent=2, ensure_ascii=False))

INPUT: Existing Neo4j graph

OUTPUT: DISCOVERED SCHEMA
{
  "nodes": {
    "Area": {
      "properties": {
        "city_name": "str",
        "dataset_id": "str",
        "name": "str",
        "name_key": "str"
      }
    },
    "Branch": {
      "properties": {
        "Area Name": "str",
        "BM Landline No": "str",
        "Branch Address": "str",
        "Branch Code": "str",
        "Branch Landline Number": "str",
        "Branch Name": "str",
        "City Name": "str",
        "OM Landline No": "str",
        "Province": "str",
        "S.No": "str",
        "branch_code": "str",
        "dataset_id": "str"
      }
    },
    "City": {
      "properties": {
        "dataset_id": "str",
        "name": "str",
        "name_key": "str"
      }
    },
    "Province": {
      "properties": {
        "dataset_id": "str",
        "name": "str",
        "name_key": "str"
      }
    }
  },
  "relationships": [
    {
      "source": [
        "Branch"
      ],
      "relationship

### 2. llm query planer

In [ ]:
import json

def plan_query(user_question, schema):
    prompt = f"""
You are a query planner for a Meezan Bank branch search system.

AVAILABLE GRAPH SCHEMA:
{json.dumps(schema, ensure_ascii=False)}

USER QUESTION:
{user_question}

Extract the user's intent and useful search clues.

Return valid JSON only, using this structure:

{{
    "intent": "branch_search",
    "location_clues": ["Wapda Town"],
    "city": "Lahore",
    "branch_code": null,
    "requested_fields": [
        "Branch Name",
        "Branch Address",
        "Branch Landline Number"
    ],
    "needs_nearest": true,
    "ambiguities": []
}}

RULES:
1. Extract location clues exactly as the user expressed them.
2. Do NOT decide that a location clue belongs to Area, City,
   Branch Name, or Branch Address.
3. We will search location clues across multiple database fields.
4. Extract a city only when the user explicitly mentions one
   or it is clearly stated in the question.
5. Extract a branch code only when the user provides one.
6. Set needs_nearest=true when the user asks for nearest,
   closest, or similar geographical proximity.
7. Do not invent database values.
8. requested_fields must contain actual Branch properties
   from the supplied schema.
9. Include ambiguities as a list of strings.
10. Never generate Cypher.

Return JSON only. Do not include Markdown fences.
"""

    response = llm.invoke(prompt)
    content = response.content.strip()

    if content.startswith("```"):
        content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    return json.loads(content)


user_question = "Tell me nearest branch to Wapda Town in Lahore"

print("INPUT QUESTION:")
print(user_question)

query_plan = plan_query(user_question, schema)

print("\nOUTPUT: LLM SEARCH PLAN")
print(json.dumps(query_plan, indent=2, ensure_ascii=False))


INPUT QUESTION:
Tell me nearest branch to Wapda Town in Lahore

OUTPUT: LLM SEARCH PLAN
{
  "intent": "branch_search",
  "location_clues": [
    "Wapda Town"
  ],
  "city": "Lahore",
  "branch_code": null,
  "requested_fields": [
    "Branch Name",
    "Branch Address",
    "Branch Landline Number"
  ],
  "needs_nearest": true,
  "ambiguities": []
}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### 3. validate the json plan

In [15]:
def validate_plan(plan, schema):

    if plan.get("intent") != "branch_search":
        raise ValueError("Unsupported intent.")

    branch_properties = set(
        schema["nodes"].get("Branch", {}).get("properties", {})
    )

    # Validate location clues
    clues = plan.get("location_clues", [])

    if not isinstance(clues, list):
        raise ValueError("location_clues must be a list.")

    plan["location_clues"] = [
        clue.strip()
        for clue in clues
        if isinstance(clue, str) and clue.strip()
    ]

    # Validate optional city
    city = plan.get("city")

    if city is not None and (
        not isinstance(city, str) or not city.strip()
    ):
        plan["city"] = None

    # Validate optional branch code
    code = plan.get("branch_code")

    if code is not None and (
        not isinstance(code, str) or not code.strip()
    ):
        plan["branch_code"] = None

    # Validate requested output fields
    requested = plan.get("requested_fields", [])

    if not isinstance(requested, list):
        raise ValueError("requested_fields must be a list.")

    invalid_fields = set(requested) - branch_properties

    if invalid_fields:
        raise ValueError(
            f"Unknown Branch properties: {invalid_fields}"
        )

    plan["requested_fields"] = requested

    plan["limit"] = max(
        1, min(int(plan.get("limit", 10)), 50)
    )

    plan["needs_nearest"] = bool(
        plan.get("needs_nearest", False)
    )

    plan["ambiguities"] = plan.get("ambiguities", [])

    if not plan["location_clues"] and not plan["city"] and not plan["branch_code"]:
        raise ValueError("No usable search clues were extracted.")

    return plan


validated_plan = validate_plan(query_plan, schema)

print("INPUT: LLM search plan")
print(json.dumps(query_plan, indent=2, ensure_ascii=False))

print("\nOUTPUT: VALIDATED SEARCH PLAN")
print(json.dumps(validated_plan, indent=2, ensure_ascii=False))


INPUT: LLM search plan
{
  "intent": "branch_search",
  "location_clues": [
    "Wapda Town"
  ],
  "city": "Lahore",
  "branch_code": null,
  "requested_fields": [
    "Branch Name",
    "Branch Address",
    "Branch Landline Number"
  ],
  "needs_nearest": true,
  "ambiguities": [],
  "limit": 10
}

OUTPUT: VALIDATED SEARCH PLAN
{
  "intent": "branch_search",
  "location_clues": [
    "Wapda Town"
  ],
  "city": "Lahore",
  "branch_code": null,
  "requested_fields": [
    "Branch Name",
    "Branch Address",
    "Branch Landline Number"
  ],
  "needs_nearest": true,
  "ambiguities": [],
  "limit": 10
}


### 4. Retrieve branch records with their connected graph information

In [16]:
def retrieve_all_branches():

    cypher = """
    MATCH (b:Branch {dataset_id: $dataset_id})

    OPTIONAL MATCH (b)-[:LOCATED_IN]->(a:Area)
    OPTIONAL MATCH (a)-[:WITHIN_CITY]->(c:City)
    OPTIONAL MATCH (c)-[:WITHIN_PROVINCE]->(p:Province)

    RETURN
        properties(b) AS branch,
        properties(a) AS area,
        properties(c) AS city,
        properties(p) AS province

    LIMIT 5000
    """

    return graph.query(
        cypher,
        {"dataset_id": "meezan_branches"}
    )


print("INPUT: Meezan Bank graph")

all_branches = retrieve_all_branches()

print("\nOUTPUT:")
print("Total retrieved branch records:", len(all_branches))

if all_branches:
    print("\nExample record:")
    print(json.dumps(all_branches[0], indent=2, ensure_ascii=False))


INPUT: Meezan Bank graph

OUTPUT:
Total retrieved branch records: 1352

Example record:
{
  "branch": {
    "branch_code": "105",
    "BM Landline No": "021-5682682",
    "Branch Address": "Marriott Hotel, Abdullah Haroon Road, Karachi",
    "Branch Code": "105",
    "dataset_id": "meezan_branches",
    "Area Name": "Clifton",
    "Branch Landline Number": "021-5683491",
    "City Name": "Karachi",
    "S.No": "1",
    "Branch Name": "Marriott Hotel, Karachi",
    "Province": "Sindh",
    "OM Landline No": "021-35693166"
  },
  "area": {
    "name_key": "karachi|clifton",
    "city_name": "Karachi",
    "dataset_id": "meezan_branches",
    "name": "Clifton"
  },
  "city": {
    "name_key": "karachi",
    "dataset_id": "meezan_branches",
    "name": "Karachi"
  },
  "province": {
    "name_key": "sindh",
    "dataset_id": "meezan_branches",
    "name": "Sindh"
  }
}


### 5. Normalize and Compare Records

In [17]:

import re
from difflib import SequenceMatcher


def normalize_text(value):
    """
    Lowercase text, remove punctuation and normalize whitespace.
    """
    if value is None:
        return ""

    value = str(value).casefold()
    value = re.sub(r"[^a-z0-9]+", " ", value)

    return " ".join(value.split())


def score_text_match(clue, candidate):

    clue_clean = normalize_text(clue)
    candidate_clean = normalize_text(candidate)

    if not clue_clean or not candidate_clean:
        return 0.0

    # 1. Exact phrase match
    if clue_clean == candidate_clean:
        return 1.0

    # 2. Full clue appears inside candidate
    if clue_clean in candidate_clean:
        return 1.0

    # 3. Token overlap
    clue_tokens = set(clue_clean.split())
    candidate_tokens = set(candidate_clean.split())

    overlap = len(clue_tokens & candidate_tokens)
    token_score = overlap / len(clue_tokens)

    # 4. Character-level similarity
    character_score = SequenceMatcher(
        None, clue_clean, candidate_clean
    ).ratio()

    return max(
        token_score * 0.85,
        character_score * 0.70
    )


# Which database fields should be searched?
SEARCH_FIELDS = {
    "Branch Name": 1.00,
    "Branch Address": 1.00,
    "Area Name": 0.95,
    "City Name": 0.40,
    "Province": 0.30,
    "Branch Code": 0.80
}


def get_searchable_fields(record):

    branch = record.get("branch") or {}
    area = record.get("area") or {}
    city = record.get("city") or {}
    province = record.get("province") or {}

    return {
        "Branch Name": branch.get("Branch Name"),
        "Branch Code": branch.get("Branch Code"),
        "Branch Address": branch.get("Branch Address"),
        "Area Name": (
            area.get("name") or branch.get("Area Name")
        ),
        "City Name": (
            city.get("name") or branch.get("City Name")
        ),
        "Province": (
            province.get("name") or branch.get("Province")
        )
    }


def score_branch(record, location_clues):

    fields = get_searchable_fields(record)

    clue_results = []

    for clue in location_clues:

        field_scores = []

        for field_name, weight in SEARCH_FIELDS.items():

            value = fields.get(field_name)

            if not value:
                continue

            similarity = score_text_match(clue, value)

            weighted_score = similarity * weight

            if weighted_score > 0:
                field_scores.append({
                    "field": field_name,
                    "matched_value": value,
                    "similarity": round(similarity, 3),
                    "score": round(weighted_score, 3)
                })

        best_match = (
            max(field_scores, key=lambda x: x["score"])
            if field_scores else None
        )

        clue_results.append({
            "clue": clue,
            "best_match": best_match
        })

    matched_scores = [
        result["best_match"]["score"]
        for result in clue_results
        if result["best_match"]
    ]

    # Average the best match for each clue.
    # Missing clues contribute zero, rather than disappearing.
    overall_score = (
        sum(matched_scores) / len(location_clues)
        if location_clues else 0
    )

    return {
        "score": round(overall_score, 3),
        "matches": clue_results
    }


print("Similarity matching functions are ready.")


Similarity matching functions are ready.


### 6. Rank matching branches

In [18]:

def rank_branches(plan, records):

    ranked = []

    requested_city = normalize_text(plan.get("city"))
    requested_code = normalize_text(plan.get("branch_code"))

    for record in records:

        fields = get_searchable_fields(record)

        # Explicit city constraint
        if requested_city:

            actual_city = normalize_text(fields.get("City Name"))

            if actual_city != requested_city:
                continue

        # Explicit branch-code constraint
        if requested_code:

            actual_code = normalize_text(fields.get("Branch Code"))

            if actual_code != requested_code:
                continue

        match_result = score_branch(
            record,
            plan["location_clues"]
        )

        # Do not include completely unrelated records.
        if plan["location_clues"] and match_result["score"] < 0.15:
            continue

        ranked.append({
            "branch": record.get("branch") or {},
            "area": record.get("area") or {},
            "city": record.get("city") or {},
            "province": record.get("province") or {},
            "score": match_result["score"],
            "matches": match_result["matches"]
        })

    ranked.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return ranked[:plan["limit"]]


print("INPUT: Validated search plan")
print(json.dumps(validated_plan, indent=2, ensure_ascii=False))

ranked_results = rank_branches(
    validated_plan,
    all_branches
)

print("\nOUTPUT: RANKED RESULTS")
print("Total matching candidates:", len(ranked_results))

for i, item in enumerate(ranked_results, start=1):

    branch = item["branch"]

    print(f"\nCandidate {i}")
    print("Branch:", branch.get("Branch Name"))
    print("Area:", item["area"].get("name") or branch.get("Area Name"))
    print("City:", item["city"].get("name") or branch.get("City Name"))
    print("Address:", branch.get("Branch Address"))
    print("Score:", item["score"])

    print("Why it matched:")
    print(json.dumps(item["matches"], indent=2, ensure_ascii=False))


INPUT: Validated search plan
{
  "intent": "branch_search",
  "location_clues": [
    "Wapda Town"
  ],
  "city": "Lahore",
  "branch_code": null,
  "requested_fields": [
    "Branch Name",
    "Branch Address",
    "Branch Landline Number"
  ],
  "needs_nearest": true,
  "ambiguities": [],
  "limit": 10
}

OUTPUT: RANKED RESULTS
Total matching candidates: 10

Candidate 1
Branch: Wapda Town Chowk, Lahore
Area: Johar Town
City: Lahore
Address: Plot No. 429, Block-E, Main Boulevard, P.I.A Employees Co-operative Housing Society, Wapda Town Chowk, Lahore.
Score: 1.0
Why it matched:
[
  {
    "clue": "Wapda Town",
    "best_match": {
      "field": "Branch Name",
      "matched_value": "Wapda Town Chowk, Lahore",
      "similarity": 1.0,
      "score": 1.0
    }
  }
]

Candidate 2
Branch: New Garden Town, Lahore
Area: NGT
City: Lahore
Address: Commercial Flat # A, Block # 5, Ground Floor, AWAMI Flats Site 1 to 4, Usman Block, New Garden Town, Lahore
Score: 0.425
Why it matched:
[
  {
    "c

In [19]:

evidence = ranked_results

print("INPUT: Ranked candidates")
print("Candidate count:", len(evidence))

print("\nOUTPUT: FINAL RETRIEVAL EVIDENCE")

for i, item in enumerate(evidence, start=1):

    branch = item["branch"]

    print(f"\n{i}. {branch.get('Branch Name')}")
    print("Branch Code:", branch.get("Branch Code"))
    print("Area:", item["area"].get("name") or branch.get("Area Name"))
    print("City:", item["city"].get("name") or branch.get("City Name"))
    print("Province:", item["province"].get("name") or branch.get("Province"))
    print("Address:", branch.get("Branch Address"))
    print("Match score:", item["score"])


INPUT: Ranked candidates
Candidate count: 10

OUTPUT: FINAL RETRIEVAL EVIDENCE

1. Wapda Town Chowk, Lahore
Branch Code: 236
Area: Johar Town
City: Lahore
Province: Punjab
Address: Plot No. 429, Block-E, Main Boulevard, P.I.A Employees Co-operative Housing Society, Wapda Town Chowk, Lahore.
Match score: 1.0

2. New Garden Town, Lahore
Branch Code: 204
Area: NGT
City: Lahore
Province: Punjab
Address: Commercial Flat # A, Block # 5, Ground Floor, AWAMI Flats Site 1 to 4, Usman Block, New Garden Town, Lahore
Match score: 0.425

3. Allama Iqbal Town Road, Lahore
Branch Code: 212
Area: Allama Iqbal Town
City: Lahore
Province: Punjab
Address: 06-Asif Block, Main Boulevard, Allama Iqbal Town, Lahore
Match score: 0.425

4. Johar Town, Lahore
Branch Code: 214
Area: Johar Town
City: Lahore
Province: Punjab
Address: Plot # 63/R-1,Johar Town Lahore
Match score: 0.425

5. Zarrar Shaheed Road Branch
(Ex: Airport Road), Lahore
Branch Code: 220
Area: Lahore Cantt
City: Lahore
Province: Punjab
Address:

### 7. End to end Search Pipeline 

In [20]:
import json

def execute_branch_search_pipeline(user_question):

    # STEP 1: LLM QUERY PLANNING

    query_plan = plan_query(
        user_question,
        schema
    )

    plan_output = json.dumps(
        query_plan,
        indent=2,
        ensure_ascii=False
    )

    # STEP 2: VALIDATE SEARCH PLAN

    validated_plan = validate_plan(
        query_plan,
        schema
    )

    validated_output = json.dumps(
        validated_plan,
        indent=2,
        ensure_ascii=False
    )

    # STEP 3: RETRIEVE BRANCHES FROM NEO4J

    all_branches = retrieve_all_branches()

    retrieval_output = (
        f"Total branch records retrieved: {len(all_branches)}\n"
        f"Source: Neo4j\n"
        f"Dataset: meezan_branches"
    )

    # STEP 4: CROSS-FIELD MATCHING

    ranked_results = rank_branches(
        validated_plan,
        all_branches
    )

    # Matching diagnostics for developer inspection

    matching_details = []

    for index, item in enumerate(ranked_results, start=1):

        branch = item.get("branch") or {}

        matching_details.append({
            "rank": index,
            "branch_name": branch.get("Branch Name"),
            "branch_code": branch.get("Branch Code"),
            "score": item.get("score"),
            "matches": item.get("matches", [])
        })

    matching_output = json.dumps(
        matching_details,
        indent=2,
        ensure_ascii=False
    )

    # STEP 5: PREPARE RETRIEVED EVIDENCE

    branch_rows = []

    for item in ranked_results:

        branch = item.get("branch") or {}
        area = item.get("area") or {}
        city = item.get("city") or {}
        province = item.get("province") or {}

        matched_fields = []

        for match in item.get("matches", []):

            best = match.get("best_match")

            if best:
                matched_fields.append(
                    f"{match['clue']} → {best['field']}"
                )

        branch_rows.append({

            "Branch Name":
                branch.get("Branch Name"),

            "Branch Code":
                branch.get("Branch Code"),

            "Area":
                area.get("name") or branch.get("Area Name"),

            "City":
                city.get("name") or branch.get("City Name"),

            "Province":
                province.get("name") or branch.get("Province"),

            "BM Phone":
                branch.get("BM Landline No"),

            "Branch Phone":
                branch.get("Branch Landline Number"),

            "OM Phone":
                branch.get("OM Landline No"),

            "Address":
                branch.get("Branch Address"),

            "Match Score":
                item.get("score"),

            "Matched Fields":
                ", ".join(matched_fields)
        })

    evidence_df = pd.DataFrame(branch_rows)

    # STEP 6: GENERATE CUSTOMER-FRIENDLY RESPONSE

    if not ranked_results:

        answer_output = (
            "I couldn't find a matching Meezan Bank branch "
            "based on that location. Could you share another "
            "nearby landmark, area, or city so I can help you "
            "find the right branch?"
        )

    else:

        answer_prompt = f"""
You are a helpful and conversational Meezan Bank branch
information assistant.

Your job is to help customers find branch information
and answer their questions naturally.

CUSTOMER'S QUESTION:
{user_question}

RETRIEVED BRANCH INFORMATION:
{json.dumps(ranked_results, ensure_ascii=False, indent=2)}

IMPORTANT RESPONSE INSTRUCTIONS:

1. Answer like a helpful human customer-service assistant.
   Use natural, friendly and simple language.

2. Focus on what the customer actually wants to know.
   Do not explain the internal retrieval process.

3. NEVER mention:
   - Similarity scores
   - Match scores
   - Ranking algorithms
   - Cross-field matching
   - Database retrieval
   - Neo4j
   - LLM planning
   - Internal technical implementation

4. If multiple branches match the customer's request,
   present them in a clean, readable format.

5. For each relevant branch, include available information:
   - Branch name
   - Branch code
   - Area or location
   - Full address
   - Branch landline
   - BM landline
   - OM landline, if available

6. Never invent missing information.
   If a contact number or other detail is unavailable,
   simply omit it or mention that it is not available.

7. Explain why a branch is relevant using ordinary language.
   For example:
   "I found a branch in Wapda Town Chowk, and its address
   mentions Main Boulevard."

8. If the customer asks for the nearest branch, be honest.
   The available information contains addresses but no
   geographic coordinates or road-distance information.
   Therefore, you cannot establish which branch is
   physically closest.

   In that situation, explain the limitation politely
   and offer the most relevant location-based results.

9. Do not describe a branch as the nearest unless actual
   geographic distance has been calculated.

10. Avoid unnecessary technical explanations and lengthy
    introductions.

RESPONSE STYLE:

- Friendly and conversational.
- Clear and easy to understand.
- Use headings only when they improve readability.
- Use bullet points for multiple branches.
- Make contact information easy to find.
- Do not overwhelm the customer with unnecessary details.

Respond directly to the customer's question.
"""

        response = llm.invoke(answer_prompt)

        answer_output = response.content

    # STEP 7: PIPELINE STATUS

    status_output = (
        f"Search completed successfully.\n"
        f"Neo4j records retrieved: {len(all_branches)}\n"
        f"Matching candidates: {len(ranked_results)}\n"
        f"Final results displayed: {len(evidence_df)}"
    )

    # RETURN ALL PIPELINE OUTPUTS

    return {
        "answer": answer_output,
        "plan": plan_output,
        "validated_plan": validated_output,
        "retrieval": retrieval_output,
        "matching": matching_output,
        "evidence": evidence_df,
        "status": status_output
    }


### 8. gradio Interface

In [24]:
import traceback

def run_branch_search(user_question):

    if not user_question or not user_question.strip():

        return (
            "Please enter a question.",
            "{}",
            "{}",
            "Not executed",
            "[]",
            pd.DataFrame(),
            "No answer generated.",
            "Waiting for a question."
        )

    try:

        # Call the complete pipeline in one place

        results = execute_branch_search_pipeline(
            user_question
        )

        # Pass pipeline results to UI components

        return (
            results["answer"],
            results["plan"],
            results["validated_plan"],
            results["retrieval"],
            results["matching"],
            results["evidence"],
            results["answer"],
            results["status"]
        )

    except Exception as e:

        error_output = (
            f"{type(e).__name__}: {str(e)}\n\n"
            f"{traceback.format_exc()}"
        )

        return (
            "Something went wrong while searching. "
            "Please try again.",
            "{}",
            "{}",
            error_output,
            error_output,
            pd.DataFrame(),
            "Answer not generated.",
            f"Failed: {str(e)}"
        )



In [26]:
import gradio as gr
import pandas as pd


custom_css = """

/* Keep the original Gradio background and text colours */

.gradio-container label {
    font-weight: 600 !important;
    border-radius: 5px !important;
    padding: 3px 7px !important;
}

/* Keep input fields looking clean */
.gradio-container input,
.gradio-container textarea {
    border-radius: 8px !important;
}

/* Subtle orange input focus */
.gradio-container input:focus,
.gradio-container textarea:focus {
    border-color: #E8B477 !important;
    box-shadow: 0 0 0 1px rgba(217, 119, 6, 0.12) !important;
}

/* Keep accordion appearance natural */
.gradio-container .accordion > button {
    font-weight: 600 !important;
}

/* Slightly rounded components */
.gradio-container .block {
    border-radius: 10px !important;
}

"""


# ==========================================
# 4. GRADIO INTERFACE
# ==========================================

with gr.Blocks(
    title="Meezan Bank Branch Search",

    # Original Gradio Soft theme
    theme=gr.themes.Soft(),

    # Only subtle orange accents
    css=custom_css

) as demo:

    gr.Markdown(
        """
        # 🏦 Meezan Bank Branch Search

        ### AI-powered natural language branch retrieval

        Ask questions in normal language and get helpful
        branch information through a conversational assistant.
        """
    )

    # QUESTION INPUT

    with gr.Row():

        question_input = gr.Textbox(
            label="Ask your question",
            placeholder=(
                "e.g. Tell me nearest branch to Wapda Town in Lahore"
            ),
            lines=2,
            scale=5
        )

    # ACTION BUTTONS

    with gr.Row():

        search_btn = gr.Button(
            "Search Branches",
            variant="primary"
        )

        clear_btn = gr.Button(
            "Clear",
            variant="secondary"
        )

    # CUSTOMER-FACING RESPONSE

    gr.Markdown("## Branch Search Response")

    answer_output = gr.Markdown()

    # PIPELINE STATUS

    status_output = gr.Textbox(
        label="Pipeline Status",
        interactive=False
    )

    gr.Markdown("---")

    gr.Markdown("## Pipeline Execution Details")

    # STEP 1

    with gr.Accordion(
        "1. LLM Query Plan",
        open=True
    ):

        plan_output = gr.Code(
            label="Extracted Search Plan",
            language="json",
            interactive=False
        )

    # STEP 2

    with gr.Accordion(
        "2. Validated Search Plan",
        open=False
    ):

        validated_output = gr.Code(
            label="Validated Plan",
            language="json",
            interactive=False
        )

    # STEP 3

    with gr.Accordion(
        "3. Neo4j Retrieval",
        open=False
    ):

        retrieval_output = gr.Textbox(
            label="Database Retrieval Details",
            lines=3,
            interactive=False
        )

    # STEP 4

    with gr.Accordion(
        "4. Cross-field Matching & Similarity",
        open=True
    ):

        matching_output = gr.Code(
            label="Matching Details",
            language="json",
            interactive=False
        )

    # STEP 5

    with gr.Accordion(
        "5. Retrieved Evidence",
        open=True
    ):

        evidence_output = gr.Dataframe(
            label="Ranked Branch Records",
            interactive=False,
            wrap=True
        )

    # STEP 6

    with gr.Accordion(
        "6. Final LLM Response",
        open=True
    ):

        final_answer_output = gr.Markdown()

    # ======================================
    # 5. BUTTON ACTIONS
    # ======================================

    outputs = [
        answer_output,
        plan_output,
        validated_output,
        retrieval_output,
        matching_output,
        evidence_output,
        final_answer_output,
        status_output
    ]

    search_btn.click(
        fn=run_branch_search,
        inputs=question_input,
        outputs=outputs
    )

    question_input.submit(
        fn=run_branch_search,
        inputs=question_input,
        outputs=outputs
    )

    def clear_interface():

        return (
            "",
            "{}",
            "{}",
            "",
            "[]",
            pd.DataFrame(),
            "",
            ""
        )

    clear_btn.click(
        fn=clear_interface,
        inputs=[],
        outputs=[question_input] + outputs
    )


# ==========================================
# 6. LAUNCH
# ==========================================

demo.launch(
    share=False,
    debug=True
)


/tmp/ipykernel_17481/4036546370.py:45: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Keyboard interruption in main thread... closing server.


### test response generation

In [18]:

if not evidence:

    answer_text = (
        "I couldn't find a matching branch in the available "
        "database records. Please provide another location clue."
    )

else:

    answer_prompt = f"""
You are a Meezan Bank branch information assistant.

Answer the user's original question using ONLY the
retrieved database evidence below.

ORIGINAL QUESTION:
{user_question}

RETRIEVED EVIDENCE:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

IMPORTANT RULES:

1. Never invent branch information, phone numbers or addresses.
2. Explain which branch details match the user's location clue.
3. Distinguish a strong textual match from a weak or partial match.
4. Do not claim a branch is geographically nearest.
5. The database contains addresses but does not contain
   coordinates or road-distance information.
6. If the user asks for the nearest branch, clearly explain
   that geographical distance cannot currently be calculated.
7. You may identify the strongest textual candidate, but
   do not present it as the geographically nearest branch.
8. If phone information is missing, say it is unavailable.

Give a concise and helpful answer.
"""

    response = llm.invoke(answer_prompt)
    answer_text = response.content


print("INPUT: Original question")
print(user_question)

print("\nOUTPUT: FINAL ANSWER")
print(answer_text)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


INPUT: Original question
Tell me nearest branch to Wapda Town in Lahore

OUTPUT: FINAL ANSWER
**Strongest textual match**

| Detail | Information |
|--------|-------------|
| **Branch name** | Wapda Town Chowk, Lahore |
| **Branch code** | 236 |
| **Address** | Plot No. 429, Block‑E, Main Boulevard, P.I.A Employees Co‑operative Housing Society, Wapda Town Chowk, Lahore |
| **BM Landline** | 042‑35211597 |
| **Branch landline** | 042‑35211591‑94 |
| **OM Landline** | 042‑35211590 |

**Why this branch is highlighted**

- The branch name and address contain “Wapda Town Chowk”, which is an exact (100 % similarity) textual match to your location clue “Wapda Town”.  
- Its relevance score is 1.0, the highest among all retrieved records.

**Other branches**

The remaining branches (e.g., New Garden Town, Allama Iqbal Town, Johar Town, etc.) have only partial similarity to “Wapda Town” and scored 0.425, indicating weaker textual relevance.

**Note on “nearest”**

Our database provides only tex

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [19]:

import pandas as pd
from IPython.display import display

branch_rows = []

for item in evidence:

    branch = item["branch"]
    area = item["area"]
    city = item["city"]
    province = item["province"]

    matched_fields = []

    for result in item["matches"]:
        best = result.get("best_match")

        if best:
            matched_fields.append(
                f"{result['clue']} → {best['field']}"
            )

    branch_rows.append({
        "Branch Name": branch.get("Branch Name"),
        "Branch Code": branch.get("Branch Code"),
        "Area": area.get("name") or branch.get("Area Name"),
        "City": city.get("name") or branch.get("City Name"),
        "Province": province.get("name") or branch.get("Province"),
        "BM Phone": branch.get("BM Landline No"),
        "Branch Phone": branch.get("Branch Landline Number"),
        "Address": branch.get("Branch Address"),
        "Match Score": item["score"],
        "Matched Fields": ", ".join(matched_fields)
    })

evidence_df = pd.DataFrame(branch_rows).fillna("N/A")

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 250
):
    display(evidence_df)


,Branch Name,Branch Code,Area,City,Province,BM Phone,Branch Phone,Address,Match Score,Matched Fields
0,"Wapda Town Chowk, Lahore",236,Johar Town,Lahore,Punjab,042-35211597,042-35211591-94,"Plot No. 429, Block-E, Main Boulevard, P.I.A Employees Co-operative Housing Society, Wapda Town Chowk, Lahore.",1.000,Wapda Town → Branch Name
1,"New Garden Town, Lahore",204,NGT,Lahore,Punjab,042-5941471,042-5941474-77,"Commercial Flat # A, Block # 5, Ground Floor, AWAMI Flats Site 1 to 4, Usman Block, New Garden Town, Lahore",0.425,Wapda Town → Branch Name
2,"Allama Iqbal Town Road, Lahore",212,Allama Iqbal Town,Lahore,Punjab,042-37870296,042-37806860-73,"06-Asif Block, Main Boulevard, Allama Iqbal Town, Lahore",0.425,Wapda Town → Branch Name
3,"Johar Town, Lahore",214,Johar Town,Lahore,Punjab,042-35241029,042-35241021-25,"Plot # 63/R-1,Johar Town Lahore",0.425,Wapda Town → Branch Name
4,"Zarrar Shaheed Road Branch\n(Ex: Airport Road), Lahore",220,Lahore Cantt,Lahore,Punjab,042-36674861,042-36674862-66,"Khasra # 3939 / 3296, Block-B, Al-Faisal Town, Zarrar Shaheed Road, Lahore Cantt.",0.425,Wapda Town → Branch Address
5,"Model Town Link Road, Lahore",281,Model Town,Lahore,Punjab,042-35201866,042-35201862-65,"Meezan Bank Limited, 13-A, Link Road, Model Town, Lahore.",0.425,Wapda Town → Branch Name
6,"Thokar Niaz Baig Branch, Lahore",224,Canal,Lahore,Punjab,042-35453648,04235453648-49,"171-A, Ali Town, Raiwand Road, Thokar Niaz Baig, Lahore",0.425,Wapda Town → Branch Address
7,"Karim Block Branch, Lahore",284,Allama Iqbal Town,Lahore,Punjab,042-35295686,042-35295681-83,"21- Commercial Zone,Karim Block Allama Iqbal Town, Lahore""",0.425,Wapda Town → Branch Address
8,"Model Town C Block Branch, Lahore",287,Model Town,Lahore,Punjab,042-35915568,042-35844202-04,"Shop # 16-18, Ground Floor, Central Commercial Maket Area, Model Town, Lahore",0.425,Wapda Town → Branch Name
9,"Moon Market Allama Iqbal Town, Lahore",231,Allama Iqbal Town,Lahore,Punjab,042-37804645,042-35427936-40,"10, Kashmir block Main Boulaward Moon Market Allama Iqbal Town Lhr",0.425,Wapda Town → Branch Name
